# Scraping Ulasan Google Play - Bank Jago

Notebook ini melakukan scraping ulasan aplikasi Bank Jago dari Google Play Store.
Output: `data/raw/reviews.csv` (~10.000 baris).

In [ ]:
import csv
import os
import time
from google_play_scraper import reviews_all, Sort

In [ ]:
APP_ID = "com.jago.digitalBanking"
OUTPUT_FILE = "../data/raw/reviews.csv"
TARGET_COUNT = 10000
LANG = "id"
COUNTRY = "id"

In [ ]:
os.makedirs('../data/raw', exist_ok=True)

In [ ]:
print(f"Scraping {TARGET_COUNT} reviews for {APP_ID}...")
result = reviews_all(APP_ID, sleep_milliseconds=1000, lang=LANG, country=COUNTRY, sort=Sort.NEWEST)
print(f"Fetched {len(result)} reviews total")

In [ ]:
# Deduplikasi berdasarkan reviewId
id_filter = set()
unique = []
for r in result:
    if r['reviewId'] not in id_filter:
        id_filter.add(r['reviewId'])
        unique.append(r)
print(f"Unique reviews: {len(unique)}")

In [ ]:
# Simpan ke CSV (sampel pertama TARGET_COUNT)
sample = unique[:TARGET_COUNT]
with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['reviewId', 'content', 'score', 'at', 'userName'])
    writer.writeheader()
    for r in sample:
        writer.writerow({
            'reviewId': r['reviewId'],
            'content': r['content'],
            'score': r['score'],
            'at': r['at'].isoformat() if hasattr(r['at'], 'isoformat') else r['at'],
            'userName': r['userName'],
        })
print(f'Saved {len(sample)} reviews to {OUTPUT_FILE}')

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT_FILE)
print(f'Dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nScore distribution:')
print(df['score'].value_counts().sort_index())
print(f"\nPositive: {(df['score']>=4).sum()}, Neutral: {(df['score']==3).sum()}, Negative: {(df['score']<=2).sum()}")
df.head()

## Verifikasi Output

In [ ]:
# Verifikasi file exists dan valid
assert os.path.exists(OUTPUT_FILE), f'File {OUTPUT_FILE} tidak ditemukan!'
assert len(df) == len(sample), 'Jumlah baris CSV tidak sesuai dengan sample!'
print(f'OK: {OUTPUT_FILE} valid, {len(df)} rows')